# 04 — Initialization: He and Xavier

Notebook 3 left us with a precise diagnosis: a deep network's gradient vanishes or explodes because
the per-layer factor (`W · activation'`) drifts away from 1, and depth compounds the drift. We could
not reshape the activation's slope — but we *can* choose the size of the weights. This notebook turns
chapter 11's hand-wave ("the weights must break symmetry") into a precise rule, and watches a dead
ten-layer network come back to life with nothing changed but its starting scale.

**Prerequisites**
- Notebook 3 — the vanishing/exploding pathology: the per-layer factor and how depth compounds it.
- Notebooks 1–2 — the `L`-layer network (forward pass, backward pass, training loop), by hand.
- Chapter 11, NB 3 — "the initial weights must break symmetry (random, not zeros)" — which we now
  make quantitative.

**What you'll be able to do**
- Derive how a layer scales the variance of its signal, and pick the weight scale that preserves it.
- Implement **He** (for ReLU) and **Xavier/Glorot** (for tanh) initialization.
- Show the gradient go flat across depth, and a deep network train where it could not before.
- Explain why **sigmoid stays the awkward case** even with the right init.

## From diagnosis to cure

Notebook 3's mechanism: the gradient reaching an early layer is a product of one factor per layer, and
each factor is roughly `W · activation'`. When the factors are below 1 the product vanishes; above 1,
it explodes. We cannot reshape `activation'` (that is fixed by our choice of ReLU or tanh), but the
**scale of the weights `W` is entirely ours to choose**.

So the question is sharp: what weight scale keeps each layer from shrinking or growing the signal — so
that the per-layer factor stays close to 1 and the product neither dies nor blows up across depth?

## Derive the rule: variance in, variance out

Look at a single layer `y = W·x` with `n` inputs, weights drawn independently from `N(0, σ²)`, and an
input `x` whose components have variance `Var(x)`. Each output `yⱼ` is a sum of `n` independent terms
`Wⱼᵢ · xᵢ`, so its variance adds up:

`Var(y) = n · σ² · Var(x)`

For the signal to come out the same size it went in — `Var(y) = Var(x)` — we need `n · σ² = 1`, i.e.

`σ = 1/√n`     ← **Xavier / Glorot** initialization

This is right for tanh, which is roughly linear (slope ~1) near zero. **ReLU is different**: with
zero-mean pre-activations it keeps only the positive half, which halves the signal's *second moment*
`E[x²]` — the quantity He et al. track (for a zero-mean signal the second moment is the variance; ReLU
shifts the mean, so the two part ways, and we follow `E[x²]`). To keep that preserved we double the
weight variance:

`σ = √(2/n)`    ← **He** initialization

(Glorot's symmetric version balances the forward and backward passes by averaging the layer's input
and output sizes: `σ² = 2 / (n_in + n_out)`.) That is the whole idea — let's implement it and measure.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_circles
from sklearn.preprocessing import StandardScaler

from ml_course import colors, viz

viz.use_course_style()

SEED = 0


# The L-layer net is exactly notebook 3's. The only new piece is how we choose the weight SCALE:
# instead of a flat number, it depends on the layer's fan-in n (the number of inputs).
def init_params(sizes, kind, seed=SEED):
    rng = np.random.default_rng(seed)
    params = []
    for n_in, n_out in zip(sizes[:-1], sizes[1:], strict=True):
        if kind == "small":            # naive: a flat tiny scale (notebooks 1-3)
            sigma = 0.1
        elif kind == "large":          # naive: a flat large scale (the exploding case)
            sigma = 1.0
        elif kind == "xavier":         # tanh: preserve variance -> sigma = 1/sqrt(n_in)
            sigma = np.sqrt(1.0 / n_in)
        elif kind == "he":             # ReLU: compensate the halving -> sigma = sqrt(2/n_in)
            sigma = np.sqrt(2.0 / n_in)
        elif kind == "glorot":         # symmetric: sigma^2 = 2/(n_in + n_out)
            sigma = np.sqrt(2.0 / (n_in + n_out))
        params.append({"W": rng.standard_normal((n_in, n_out)) * sigma, "b": np.zeros(n_out)})
    return params


def activate(z, name):
    if name == "sigmoid":
        return 1.0 / (1.0 + np.exp(-z))
    if name == "tanh":
        return np.tanh(z)
    return np.maximum(0.0, z)  # relu


def deriv_from_output(a, name):
    if name == "sigmoid":
        return a * (1.0 - a)
    if name == "tanh":
        return 1.0 - a ** 2
    return (a > 0).astype(float)  # relu


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def forward(params, X, hidden):
    acts, a, last = [X], X, len(params) - 1
    for i, layer in enumerate(params):
        z = a @ layer["W"] + layer["b"]
        a = softmax(z) if i == last else activate(z, hidden)
        acts.append(a)
    return acts


def backward(params, acts, y, hidden):
    n = len(y)
    y_onehot = np.zeros_like(acts[-1])
    y_onehot[np.arange(n), y] = 1.0
    d = (acts[-1] - y_onehot) / n
    grads = [None] * len(params)
    for i in reversed(range(len(params))):
        grads[i] = {"W": acts[i].T @ d, "b": d.sum(axis=0)}
        if i > 0:
            d = (d @ params[i]["W"].T) * deriv_from_output(acts[i], hidden)
    return grads


# Same deep stack as notebook 3: 10 hidden layers of width 16, on standardized circles.
X, y = make_circles(n_samples=400, noise=0.1, factor=0.4, random_state=SEED)
X = StandardScaler().fit_transform(X)
sizes = [2] + [16] * 10 + [2]


def gradient_rms(hidden, kind):
    params = init_params(sizes, kind)
    grads = backward(params, forward(params, X, hidden), y, hidden)
    return np.array([np.sqrt(np.mean(g["W"] ** 2)) for g in grads])


def activation_std(hidden, kind):
    params = init_params(sizes, kind)
    acts = forward(params, X, hidden)
    return np.array([acts[i].std() for i in range(1, len(params))])  # hidden layers only


print("He scale for n=16:    ", round(np.sqrt(2 / 16), 4))
print("Xavier scale for n=16:", round(np.sqrt(1 / 16), 4))

## The test: re-run notebook 3's measurement

We measure the per-layer gradient exactly as in notebook 3 — its root-mean-square at each layer — but
now we compare the naive scales (small, large) against the variance-preserving ones (He for ReLU,
Xavier for tanh).

In [ ]:
g_relu_small = gradient_rms("relu", "small")
g_relu_large = gradient_rms("relu", "large")
g_relu_he = gradient_rms("relu", "he")
g_tanh_xavier = gradient_rms("tanh", "xavier")
g_sig_small = gradient_rms("sigmoid", "small")
g_sig_xavier = gradient_rms("sigmoid", "xavier")

print(f"ReLU + small : input {g_relu_small[0]:.1e}  (alive but tiny)")
print(f"ReLU + large : input {g_relu_large[0]:.1e}  (exploding)")
print(f"ReLU + He    : {g_relu_he.min():.2f} to {g_relu_he.max():.2f}  (flat and healthy)")
print(f"tanh + Xavier: {g_tanh_xavier.min():.1e} to {g_tanh_xavier.max():.1e}  (flat and healthy)")

In [ ]:
layers = np.arange(1, len(sizes))  # 11 weight matrices

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Before: notebook 3's naive init.
ax0.plot(layers, g_sig_small, "o-", color=colors.COLORS["model"], lw=2, label="sigmoid + small")
ax0.plot(layers, g_relu_large, "s-", color=colors.COLORS["error"], lw=2, label="ReLU + large")
ax0.set_yscale("log")
ax0.set_xlabel("layer (1 = closest to input)")
ax0.set_ylabel("gradient RMS (log scale)")
ax0.set_title("Naive init: vanish or explode")
ax0.legend()

# After: variance-preserving init (+ the sigmoid caveat).
ax1.plot(layers, g_relu_he, "o-", color=colors.COLORS["correct"], lw=2, label="ReLU + He")
ax1.plot(layers, g_tanh_xavier, "s-", color=colors.COLORS["class_a"], lw=2, label="tanh + Xavier")
ax1.plot(layers, g_sig_xavier, "x--", color=colors.COLORS["error"], lw=1.8,
         label="sigmoid + Xavier (still bends)")
ax1.set_yscale("log")
ax1.set_xlabel("layer (1 = closest to input)")
ax1.set_title("Variance-preserving init: flat")
ax1.legend()

fig.tight_layout()
plt.show()

**Read the figure.** Left, notebook 3's pathology: the sigmoid+small gradient falls off a cliff toward
the input layer, the ReLU+large gradient climbs away. Right, the cure: **ReLU + He** holds the gradient
flat across all ten layers (it stays around `0.03`–`0.22`, no collapse, no blow-up — and the `0.22` is
the softmax output layer; the hidden layers sit in an even tighter band), and **tanh + Xavier** does
the same. Same network, same depth — only the starting scale changed. (One curve still bends downward
on the right: sigmoid, even with Xavier. We come back to it.)

## Why it works

The scale is doing exactly what we derived. He's `√(2/n)` makes each ReLU layer's per-layer factor land
near 1 — the `2` cancels ReLU's variance-halving — so the product across depth stays near 1 instead of
shrinking to `1e-13` or growing to `1e3`. Xavier's `1/√n` does the same for tanh, whose near-linear
middle behaves like the plain `y = Wx` we analyzed. The compounding that destroyed the gradient in
notebook 3 now compounds a number close to 1, which stays close to 1.

In [ ]:
s_relu_small = activation_std("relu", "small")
s_relu_large = activation_std("relu", "large")
s_relu_he = activation_std("relu", "he")
hidden_layers = np.arange(1, 11)

print(f"ReLU+He  activation std: layer 1 {s_relu_he[0]:.2f} -> layer 10 {s_relu_he[-1]:.2f}"
      f"  (preserved)")
print(f"ReLU+small activation std: layer 1 {s_relu_small[0]:.2f} -> layer 10 {s_relu_small[-1]:.2f}"
      f"  (starved)")

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(hidden_layers, s_relu_he, "o-", color=colors.COLORS["correct"], lw=2, label="ReLU + He")
ax.plot(hidden_layers, s_relu_small, "s-", color=colors.COLORS["muted"], lw=2, label="ReLU + small")
ax.plot(hidden_layers, s_relu_large, "^-", color=colors.COLORS["error"], lw=2, label="ReLU + large")
ax.set_yscale("log")
ax.set_xlabel("hidden layer (1 = first)")
ax.set_ylabel("activation spread — std (log scale)")
ax.set_title("Forward signal across depth: He preserves it")
ax.legend()
plt.show()

**Read the figure.** The same story, seen on the *forward* pass. **He** (teal) keeps the activation
spread roughly constant down the stack — about `0.69` at the first hidden layer, still `0.53` at the
tenth — so every layer receives a signal it can work with. **Small** weights (grey) starve the deep
layers: the spread decays toward zero, and a layer whose inputs are all ~0 has nothing to learn from.
**Large** weights (coral) blow the spread up. Preserving the variance — neither starving nor saturating
— is the whole game, forward and backward alike.

## The awkward case: sigmoid

tanh and ReLU are (near) zero-centered — tanh outputs live around 0, ReLU's kept half around a small
positive mean. **Sigmoid is not**: its outputs live in `(0, 1)` with a mean near `0.5`. That nonzero
mean is injected at every layer and accumulates, so the clean "variance in = variance out" rule no
longer holds — and Xavier, derived for the zero-centered case, cannot fully save it.

In [ ]:
ratio_tanh = g_tanh_xavier[-1] / g_tanh_xavier[0]
ratio_sig = g_sig_xavier[-1] / g_sig_xavier[0]
print("Same Xavier init, output-layer gradient / input-layer gradient:")
print(f"  tanh    : {ratio_tanh:6.1f}      (~1 would be perfectly flat; this is essentially flat)")
print(f"  sigmoid : {ratio_sig:.1e}   (still vanishing across depth)")
print(f"\nsigmoid input-layer gradient: small init {g_sig_small[0]:.1e}"
      f" -> Xavier {g_sig_xavier[0]:.1e}")
print("  (Xavier improves it by orders of magnitude, but a healthy value is ~1e-2.)")

**Read the result.** The *same* Xavier initialization that flattens tanh leaves sigmoid still dying:
tanh's gradient is essentially constant from output to input, while sigmoid's still shrinks by seven
orders of magnitude. Xavier *improves* sigmoid — from `1e-13` (small init) up to `1e-9` — but cannot
*preserve* it; a healthy input-layer gradient would be near `1e-2`. (Glorot's paper notes sigmoid needs
a larger gain to compensate its nonzero mean.) This is the deeper reason — beyond the saturation we saw
in notebook 3 — that **tanh and ReLU displaced sigmoid as the default hidden activation.** We are not
rescuing sigmoid here; we are showing precisely why it is avoided.

## The payoff that matters: does the deep net now train?

A flat gradient is the means; the end is a network that actually learns. We train an **eight-hidden-layer**
network by hand — the same gradient-descent loop from notebooks 1 and 3 — comparing the naive scales
against the matched variance-preserving ones. (Eight layers so the pathology bites; on circles, which a
shallow net handled easily.)

In [ ]:
deep = [2] + [16] * 8 + [2]


def train_accuracy(hidden, kind, lr=0.3, epochs=300, batch=64, seed=SEED):
    params = init_params(deep, kind, seed)
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    for _ in range(epochs):
        order = rng.permutation(n)
        for start in range(0, n, batch):
            b = order[start:start + batch]
            grads = backward(params, forward(params, X[b], hidden), y[b], hidden)
            for i in range(len(params)):
                params[i]["W"] -= lr * grads[i]["W"]
                params[i]["b"] -= lr * grads[i]["b"]
    return (forward(params, X, hidden)[-1].argmax(1) == y).mean()


results = {
    "ReLU\nsmall": train_accuracy("relu", "small"),
    "ReLU\nlarge": train_accuracy("relu", "large"),
    "ReLU\nHe": train_accuracy("relu", "he"),
    "tanh\nsmall": train_accuracy("tanh", "small"),
    "tanh\nXavier": train_accuracy("tanh", "xavier"),
}
for name, acc in results.items():
    print(f"{name.replace(chr(10), ' '):14}: train accuracy {acc:.3f}")

In [ ]:
names = list(results.keys())
accs = [results[n] for n in names]
# Naive inits in muted/coral, matched (He/Xavier) in teal.
bar_colors = [colors.COLORS["muted"], colors.COLORS["error"], colors.COLORS["correct"],
              colors.COLORS["muted"], colors.COLORS["correct"]]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(names, accs, color=bar_colors, edgecolor=colors.COLORS["text"], linewidth=0.6)
for bar, acc in zip(bars, accs, strict=True):
    ax.text(bar.get_x() + bar.get_width() / 2, acc, f"{acc:.3f}",
            ha="center", va="bottom", color=colors.COLORS["text"])
ax.axhline(0.5, color=colors.COLORS["muted"], ls="--", lw=1, label="chance (0.5)")
ax.set_ylim(0, 1.08)
ax.set_ylabel("training accuracy")
ax.set_title("An 8-layer network: matched init turns chance into 100%")
ax.legend()
plt.show()

**Read the figure.** An eight-layer network is **hopeless with naive weights** — stuck at chance (0.5)
whether the weights are too small (gradient vanishes) or too large (gradient explodes). Switch to the
**matched variance-preserving init** — He for the ReLU network, Xavier for the tanh network — and the
*same* architecture trains to **100%**. Notebook 3's diagnosis, cured: not by changing the model, the
data, or the optimizer, but by choosing the initial scale.

## What you built

- You turned chapter 11's "break symmetry" into a precise rule: a layer scales variance by
  `Var(Wx) = n · Var(W) · Var(x)`, so the variance-preserving scale is `σ = 1/√n`.
- You derived and implemented **Xavier** (`1/√n`, for tanh) and **He** (`√(2/n)`, for ReLU — the `2`
  compensating ReLU's halving).
- You watched the gradient go **flat** across ten layers and an eight-layer network **train to 100%**
  where naive init left it at chance.
- You saw **why sigmoid stays the awkward case** — not zero-centered, so even Xavier cannot preserve it.

## Where this goes next

Initialization sets the variance right **at the start**. But the weights move during training, and the
nice balance can drift away. **Normalization** (notebook 6) keeps the signal healthy *during* training,
re-centering and re-scaling each layer as it goes — the same variance-control idea, applied at every
step rather than once. First, though, **notebook 5 — dropout**: a different lever entirely, aimed at
overfitting rather than signal flow.

## Your turn

1. **(warm-up)** Apply He to the 10-layer ReLU stack (`gradient_rms("relu", "he")`) and confirm the
   gradient RMS is flat — then compare it to `gradient_rms("relu", "small")` from notebook 3's vanish.
2. **(core)** Initialize a **ReLU** net with **Xavier** (dropping the factor of 2) and measure its
   gradient RMS — by what factor does it shrink versus He? (On this short, easy net the optimizer often
   still trains despite the mismatch; the gap matters more as depth and difficulty grow.)
3. **(reach)** Initialize a deep net with He weights but set every **bias** to a large constant (say 5)
   instead of 0. Does the signal still flow, or does the bias wreck it? (Initialization is about more
   than the weights' standard deviation.)

## References

- Glorot, X., & Bengio, Y. (2010). Understanding the difficulty of training deep feedforward neural
  networks. *Proceedings of Machine Learning Research* 9:249–256 (Xavier / Glorot initialization).
- He, K., Zhang, X., Ren, S., & Sun, J. (2015). Delving deep into rectifiers: surpassing human-level
  performance on ImageNet classification. *ICCV*. https://doi.org/10.1109/ICCV.2015.123
  (arXiv:1502.01852) — He initialization.
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*, chapter 8. MIT Press.

*Previous:* **12.3 — vanishing & exploding gradients.**  *Next:* **12.5 — dropout.**